In [0]:
import os

repo = "/tmp/Impute_then_Regress"
python_dir = f"{repo}/python"
results_dir = f"{repo}/results"

os.makedirs(results_dir, exist_ok=True)

print("Repository:", repo)
print("Python directory:", python_dir)
print("Results directory:", results_dir)

Repository: /tmp/Impute_then_Regress
Python directory: /tmp/Impute_then_Regress/python
Results directory: /tmp/Impute_then_Regress/results


In [0]:
import sys
import platform
import numpy
import pandas
import scipy
import sklearn
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Platform: Linux-6.6.144.1-1.azl3-x86_64-with-glibc2.39
NumPy: 2.3.4
pandas: 2.3.3
SciPy: 1.16.3
scikit-learn: 1.7.2
PyTorch: 2.11.0+cu130
CUDA available: True


In [0]:
from pathlib import Path

launch_file = Path(
    "/tmp/Impute_then_Regress/python/launch_all.py"
)

text = launch_file.read_text()

if "n_jobs = 40" in text:
    text = text.replace("n_jobs = 40", "n_jobs = 1")
    launch_file.write_text(text)
    print("Changed n_jobs from 40 to 1.")
elif "n_jobs = 1" in text:
    print("n_jobs is already set to 1.")
else:
    print("Inspect launch_all.py because the expected setting was not found.")

Changed n_jobs from 40 to 1.


In [0]:
for number, line in enumerate(
    launch_file.read_text().splitlines(),
    start=1
):
    if "n_jobs" in line:
        print(number, line)
        

21 n_jobs = 1
180         'n_jobs': n_jobs}


In [0]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

torch.set_num_threads(1)

print("PyTorch threads:", torch.get_num_threads())

PyTorch threads: 1


In [0]:
import subprocess
import os

python_dir = "/tmp/Impute_then_Regress/python"

environment = os.environ.copy()
environment["OMP_NUM_THREADS"] = "1"
environment["MKL_NUM_THREADS"] = "1"
environment["OPENBLAS_NUM_THREADS"] = "1"
environment["NUMEXPR_NUM_THREADS"] = "1"

process = subprocess.run(
    [
        sys.executable,
        "launch_all.py",
        "--link",
        "square",
        "MCAR",
    ],
    cwd=python_dir,
    env=environment,
    text=True,
    capture_output=False,
)

print("Exit code:", process.returncode)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:256)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:758)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:462)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:462)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Minimal reproduction of the main experiment (MCAR, square link).
# Same data generator, estimators and evaluation as launch_all.py,
# but reduced scope: 1 hyperparameter setting per method, 1 train size, 3 repetitions.
import os, sys, subprocess, textwrap

python_dir = "/tmp/Impute_then_Regress/python"

N_ITER = 3            # paper: 10
N_SIZES = [20000, 100000]  # paper Figure 4 uses n_train = 100,000; 20,000 kept to show the trend
N_JOBS = min(18, os.cpu_count())

script = textwrap.dedent(f'''
import pandas as pd
from run_all import run

default_values = {{'n_features': 50, 'missing_rate': 0.5, 'prop_latent': 0.3,
                  'snr': 10, 'masking': 'MCAR', 'prop_for_masking': None,
                  'link': 'square', 'curvature': 1}}
data_descs = pd.DataFrame([default_values])

# One fixed configuration per method (paper: grid search, best chosen on validation set)
mlp = dict(n_epochs=1000, batch_size=100, lr=1e-3, weight_decay=1e-5,
           early_stopping=True, optimizer='adam', mlp_depth=2,
           width_factor=5, init_type='uniform', verbose=False)

methods_params = [
    {{'method': 'BayesPredictor', 'order0': False}},
    {{'method': 'GBRT', 'n_iter_no_change': 10, 'max_leaf_nodes': 200,
      'max_iter': 300, 'min_samples_leaf': 20}},
    {{'method': 'meanMLPPytorch_mask', 'add_mask': True, 'imputation_type': 'mean', **mlp}},
    {{'method': 'MICEMLPPytorch_mask', 'add_mask': True, 'imputation_type': 'MICE', **mlp}},
    {{'method': 'NeuMiss_uniform_', 'mode': 'shared', 'depth': 20,
      'residual_connection': True, 'add_mask': False, **mlp}},
]

run(n_iter={N_ITER}, n_sizes={N_SIZES}, n_test=10000, n_val=10000, mdm='MCAR',
    data_descs=data_descs, methods_params=methods_params,
    filename='MCAR_square_minimal', n_jobs={N_JOBS})
''')

script_path = f"{python_dir}/run_minimal.py"
with open(script_path, "w") as f:
    f.write(script)

env = os.environ.copy()
for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    env[v] = "1"

# Patch np.Inf -> np.inf (removed in NumPy 2.0) in the repo's pytorchtools.py
pt_path = f"{python_dir}/pytorchtools.py"
with open(pt_path) as f:
    pt_src = f.read()
if "np.Inf" in pt_src:
    with open(pt_path, "w") as f:
        f.write(pt_src.replace("np.Inf", "np.inf"))
    print("Patched pytorchtools.py: np.Inf -> np.inf")

print(f"Tasks: {5 * N_ITER} (each fits all sizes {N_SIZES}), parallel jobs: {N_JOBS}")
proc = subprocess.run([sys.executable, "run_minimal.py"], cwd=python_dir, env=env, text=True)
print("Exit code:", proc.returncode)

Tasks: 15 (each fits all sizes [20000, 100000]), parallel jobs: 18


/opt/databricks-environments/databricks-ai/lib/python3.12/site-packages/sklearn/experimental/enable_hist_gradient_boosting.py:18: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(
100%|██████████| 15/15 [00:00<00:00, 5065.58it/s]
/opt/databricks-environments/databricks-ai/lib/python3.12/site-packages/sklearn/experimental/enable_hist_gradient_boosting.py:18: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(
/opt/databricks-environments/databricks-ai/lib/python3.12/site-packages/sklearn/experimental/enable_hist_gradient_boosting.py:18: UserWarning: Since version 1.0, it is not needed to import enable_

method: GBRT, dim: (20000, 50), it: 2
method: GBRT, dim: (100000, 50), it: 2
method: GBRT, dim: (20000, 50), it: 0
method: GBRT, dim: (100000, 50), it: 0
method: GBRT, dim: (20000, 50), it: 1
method: GBRT, dim: (100000, 50), it: 1
method: meanMLPPytorch_mask, dim: (20000, 50), it: 1
method: meanMLPPytorch_mask, dim: (100000, 50), it: 1
method: meanMLPPytorch_mask, dim: (20000, 50), it: 2
method: meanMLPPytorch_mask, dim: (100000, 50), it: 2
method: meanMLPPytorch_mask, dim: (20000, 50), it: 0
method: meanMLPPytorch_mask, dim: (100000, 50), it: 0
method: MICEMLPPytorch_mask, dim: (20000, 50), it: 2
method: MICEMLPPytorch_mask, dim: (100000, 50), it: 2
method: MICEMLPPytorch_mask, dim: (20000, 50), it: 0
method: MICEMLPPytorch_mask, dim: (100000, 50), it: 0
method: MICEMLPPytorch_mask, dim: (20000, 50), it: 1
method: MICEMLPPytorch_mask, dim: (100000, 50), it: 1
method: NeuMiss_uniform_, dim: (20000, 50), it: 0
method: NeuMiss_uniform_, dim: (100000, 50), it: 0
method: NeuMiss_uniform_, 

In [0]:
import pandas as pd

res = pd.read_csv("/tmp/Impute_then_Regress/results/MCAR_square_minimal.csv")

# Paper metric: test R2 minus Bayes-predictor R2 on the same repetition
bayes = res[res.method == "BayesPredictor"][["iter", "n", "R2_test"]].rename(columns={"R2_test": "R2_bayes"})
res = res.merge(bayes, on=["iter", "n"])
res["R2_difference_from_Bayes"] = res["R2_test"] - res["R2_bayes"]

summary = (res.groupby(["method", "n"])
              .agg(n_repetitions=("iter", "nunique"),
                   R2_test_mean=("R2_test", "mean"),
                   R2_test_std=("R2_test", "std"),
                   R2_difference_from_Bayes_mean=("R2_difference_from_Bayes", "mean"),
                   R2_difference_from_Bayes_std=("R2_difference_from_Bayes", "std"))
              .sort_values(["n", "R2_difference_from_Bayes_mean"], ascending=[True, False])
              .reset_index())

# Check: every method/training-size combination has exactly 3 repetitions
assert (summary["n_repetitions"] == 3).all(), summary[["method", "n", "n_repetitions"]]
print("All method/training-size combinations have 3 repetitions.")

display(summary.round(4))

# Per-repetition GBRT values (GBRT early stopping uses a random internal validation split)
display(res[res.method == "GBRT"][["iter", "n", "R2_test"]].sort_values(["n", "iter"]))

method,n,R2_test_mean,R2_test_std,R2_rel_bayes_mean,R2_rel_bayes_std
BayesPredictor,20000,0.8967786250787859,0.0020491661274389342,0.0,0.0
MICEMLPPytorch_mask,20000,0.7562955108034147,0.005016893450164683,-0.14048311427537133,0.0069161008324334005
NeuMiss_uniform_,20000,0.5444335694260788,0.031781150900688185,-0.35234505565270724,0.03294229273019356
GBRT,20000,0.5236247795422259,0.03875508821378899,-0.37315384553656,0.03897470190723753
meanMLPPytorch_mask,20000,0.49233494558402974,0.02867766902243542,-0.4044436794947563,0.02786364839246725
BayesPredictor,100000,0.8967786250787859,0.0020491661274389342,0.0,0.0
NeuMiss_uniform_,100000,0.8310777295474727,0.00510923215872684,-0.0657008955313132,0.005705581557863242
MICEMLPPytorch_mask,100000,0.8306624888205162,0.0012616206838723934,-0.06611613625826973,0.002995078450132267
meanMLPPytorch_mask,100000,0.648460255589992,0.018782868379494604,-0.24831836948879402,0.01863521981808466
GBRT,100000,0.6248689937299458,0.0334664684055304,-0.2719096313488402,0.03344530691508101
